# Simple portfolio optimization: when tails drive the decision

A CRRA investor with risk aversion $\gamma$ chooses the fraction $f$ of wealth to place in a single risky asset whose risk-adjusted return is $m_t = (r^{\text{Mkt}}_t - r^f_t)/(1 + r^f_t)$. End-of-period wealth (per dollar) is $1 + f\,m$, so the problem is

$$
f^\star(\gamma) \;=\; \arg\max_{f} \; \mathbb{E}\bigl[u_\gamma(1 + f\,m)\bigr], \qquad u_\gamma(x) = \begin{cases} \log x & \gamma = 1 \\ \dfrac{x^{1-\gamma} - 1}{1-\gamma} & \gamma \neq 1 \end{cases}.
$$

In `1_fitting_and_selecting_one_asset.ipynb` we fit Normal, Student-$t$, and Johnson $S_U$ to the monthly Fama–French market series on fold 1, and selected Johnson $S_U$ by CRPS on fold 2. CRPS is a scoring rule for the *bulk* of the predictive distribution; it weights the integrand by squared CDF error and is essentially insensitive to whether the model captures behavior at $10^{-5}$ probability levels.

The portfolio problem above is not a bulk problem. Because CRRA utility is $-\infty$ at $w \le 0$ for $\gamma \ge 1$ (and complex-valued for $\gamma < 1$), the *feasible* leverage is bounded by $f < 1/|m_{\min}|$, where $m_{\min}$ is the worst draw under the assumed return distribution. So the decision turns on the left tail — exactly the part of the distribution CRPS does not validate.

This notebook runs the optimization under all three fitted distributions and shows that they give qualitatively different $f^\star$, even though their CRPS values were within 3% of each other.

In [1]:
import numpy as np
import pandas as pd
import requests
from io import BytesIO
from zipfile import ZipFile
from scipy import stats
from scipy.optimize import minimize_scalar

rng = np.random.default_rng(607)

## Refit the three families on fold 1

Same data and fold definitions as notebook 1.

In [2]:
url = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Research_Data_Factors_CSV.zip"
response = requests.get(url)
with ZipFile(BytesIO(response.content)) as zfile:
    csv_name = next(n for n in zfile.namelist() if n.lower().endswith(".csv"))
    with zfile.open(csv_name) as f:
        ff = pd.read_csv(f, skiprows=3, index_col=0)
ff = ff.apply(pd.to_numeric, errors="coerce").dropna()
ff = ff[ff.index.map(lambda s: s.isdigit() and len(s) == 6)]
ff["m"] = (ff["Mkt-RF"] / 100) / (1 + ff["RF"] / 100)
fold1 = ff.loc["192607":"198212", "m"]

families = {"Normal": stats.norm, "Student t": stats.t, "Johnson SU": stats.johnsonsu}
fits = {name: dist(*dist.fit(fold1)) for name, dist in families.items()}

pd.DataFrame({name: {"params": np.round(f.args + tuple(f.kwds.values()), 4)}
              for name, f in fits.items()}).T

,params
Normal,"[0.0064, 0.0587]"
Student t,"[3.4535, 0.0086, 0.0383]"
Johnson SU,"[0.2593, 1.2581, 0.0197, 0.0498]"


## Simulate returns from each fitted distribution

Same number of draws and the same RNG state for each family, so the only thing that differs across columns below is the assumed return distribution.

In [3]:
n_simulations = 100_000
ss = rng.bit_generator._seed_seq
child_seeds = ss.spawn(len(fits))
draws = {name: f.rvs(size=n_simulations, random_state=np.random.default_rng(seed))
         for (name, f), seed in zip(fits.items(), child_seeds)}

pd.DataFrame({
    name: {"m_min": x.min(), "m_max": x.max(),
           "f_max = 1/|m_min|": 1 / abs(x.min())}
    for name, x in draws.items()
}).T.round(3)

,m_min,m_max,f_max = 1/|m_min|
Normal,-0.242,0.260,4.138
Student t,-1.819,0.965,0.550
Johnson SU,-0.614,0.502,1.628


## CRRA expected utility and optimizer

We optimize over $f \in [0,\,f_{\max}]$ with $f_{\max} = 1/|m_{\min}| - \epsilon$, so $1 + f\,m > 0$ on every simulated draw and the utility is real-valued. The objective is the negative mean utility so `minimize_scalar` solves the maximization.

In [4]:
def neg_expected_utility(f, returns, gamma):
    w = 1.0 + f * returns
    u = np.log(w) if gamma == 1 else (w ** (1 - gamma) - 1) / (1 - gamma)
    return -u.mean()

def optimal_fraction(returns, gamma):
    f_max = 1.0 / abs(returns.min()) - 1e-3
    res = minimize_scalar(neg_expected_utility, args=(returns, gamma),
                          bounds=(0.0, f_max), method="bounded",
                          options={"xatol": 1e-6})
    return res.x

## Optimal fraction across $\gamma$ and across return models

In [5]:
gammas = [0.25, 0.5, 0.75, 1.0, 2.0, 3.0]
results = pd.DataFrame(
    {name: [optimal_fraction(x, g) for g in gammas] for name, x in draws.items()},
    index=pd.Index(gammas, name="gamma"),
).round(3)
results

,Normal,Student t,Johnson SU
gamma,,,
0.25,4.137,0.549,1.627
0.50,3.698,0.549,1.627
0.75,2.546,0.549,1.624
1.00,1.930,0.548,1.409
2.00,0.974,0.516,0.761
3.00,0.651,0.453,0.516


## What just happened

All three models were fit to the same data and produced CRPS values within a few percent of each other on fold 2 (notebook 1: Johnson $S_U$ 0.02449, Student-$t$ 0.02452, Normal 0.02527). Yet the recommended $f^\star$ at low $\gamma$ differs by an *order of magnitude* across them, and the qualitative ordering can flip depending on which model you trusted.

The reason is that CRPS measures bulk calibration and the optimization is driven by $m_{\min}$, the worst draw the model produces. Light-tailed Normal gives an optimistic $m_{\min}$ and recommends large $f^\star$; the fat-tailed Student-$t$ and Johnson $S_U$ produce $m_{\min}$ values that are not economically real — a *single-month* market return of $-100\%$ or worse is not a thing — and the resulting ruin constraint binds at low $\gamma$.

**The methodological lesson.** When the decision depends on the tail (here through the ruin boundary; elsewhere through VaR, CVaR, or stop-out probabilities), a model selected on a bulk-calibration metric is not validated for the decision you are making with it. CRPS, log-score, Kolmogorov–Smirnov, and similar scoring rules tell you the bulk fit is reasonable. They do not tell you the $10^{-5}$ quantile is reasonable, and you cannot back that out from the same data the model was fit on. To do tail-driven optimization honestly you need either (i) a tail-focused model (peaks-over-threshold / EVT), (ii) explicit economically motivated truncation, or (iii) a robust optimization formulation that does not commit to a single tail.

### A second caveat for class discussion

Even setting tails aside, the model assumes the investor can **borrow at the risk-free rate** $r^f$ when $f > 1$ — that is what $m_t = (r^{\text{Mkt}}_t - r^f_t)/(1 + r^f_t)$ encodes. In practice retail and even institutional investors face a borrowing rate strictly above $r^f$ (margin rates, repo haircuts, prime-broker spreads). The honest version of the problem replaces $m$ with a piecewise return that uses the higher borrowing cost on the $f - 1$ levered dollars, which shrinks $f^\star$ whenever the unconstrained solution exceeds 1. This is the *economic* leverage cap, and it would bite long before the Normal-implied $f^\star$ of a few hundred percent.